In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import cv2
import numpy as np
from PIL import Image
from torchvision import models, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [2]:
image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=16000,
    n_fft=1024,
    hop_length=512,
    n_mels=64
)

db_transform = torchaudio.transforms.AmplitudeToDB()


In [3]:
class ImageEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        base = models.resnet18(weights="IMAGENET1K_V1")
        self.features = nn.Sequential(*list(base.children())[:-1])
        self.fc = nn.Linear(512, 256)

    def forward(self, x):
        x = self.features(x)
        x = x.squeeze(-1).squeeze(-1)
        return self.fc(x)


In [4]:
class AudioEncoderTemporal(nn.Module):
    def __init__(self):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d((2, 2)),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d((2, 2))
        )

        self.lstm = nn.LSTM(
            input_size=64 * 16,
            hidden_size=128,
            bidirectional=True,
            batch_first=True
        )

        self.fc = nn.Linear(256, 256)

    def forward(self, x):
        # x: [B,1,64,T]
        x = self.cnn(x)              # [B,64,16,T']
        x = x.permute(0, 3, 1, 2)    # [B,T',64,16]
        x = x.reshape(x.size(0), x.size(1), -1)

        x, _ = self.lstm(x)
        x = x.mean(dim=1)

        return self.fc(x)


In [5]:
class FusionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(512, 256),   # fc.0
            nn.ReLU(),             # fc.1
            nn.Dropout(0.3),       # fc.2  ← THIS MUST EXIST
            nn.Linear(256, 8)      # fc.3
        )

    def forward(self, img, aud):
        x = torch.cat([img, aud], dim=1)
        return self.fc(x)


In [6]:
image_encoder = ImageEncoder().to(device)
audio_encoder = AudioEncoderTemporal().to(device)
fusion_model  = FusionModel().to(device)

ckpt = torch.load("model.pth", map_location=device)

image_encoder.load_state_dict(ckpt["image"])
audio_encoder.load_state_dict(ckpt["audio"])
fusion_model.load_state_dict(ckpt["fusion"])

image_encoder.eval()
audio_encoder.eval()
fusion_model.eval()

print("✅ Model loaded successfully")


✅ Model loaded successfully


C:\Users\arnav\AppData\Local\Temp\ipykernel_16300\2995223805.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load("model.pth", map_location=device)


In [7]:
def extract_frame(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError("Cannot open video")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.set(cv2.CAP_PROP_POS_FRAMES, total_frames // 2)

    ret, frame = cap.read()
    cap.release()

    if not ret:
        raise RuntimeError("Failed to read frame")

    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    return Image.fromarray(frame)


In [8]:
import os
print(os.getcwd())
print(os.listdir())


C:\Users\arnav\Project_audio_visual
['.ipynb_checkpoints', 'audio_temporal.pth', 'DATA', 'data_upload.ipynb', 'model.pth', 'Notebook_0_env_check.ipynb', 'notebook_1_5.ipynb', 'Notebook_1_dataset_loader.ipynb', 'Notebook_2_image_model.ipynb', 'Notebook_3_5_multiframe_video.ipynb', 'Notebook_3_audio_model.ipynb', 'Notebook_4_multimodal.ipynb', 'Notebook_5_training_evaluation.ipynb', 'test_video.mp4']


In [9]:
def extract_audio_from_video(video_path):
    waveform, sr = torchaudio.load(video_path)

    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    if sr != 16000:
        waveform = torchaudio.functional.resample(
            waveform, sr, 16000
        )

    return waveform


In [10]:
@torch.no_grad()
def predict_from_video(video_path):
    # ----- IMAGE -----
    img = extract_frame(video_path)
    img = image_transform(img).unsqueeze(0).to(device)

    # ----- AUDIO -----
    wav = extract_audio_from_video(video_path)

    mel = mel_transform(wav)
    mel = db_transform(mel)

    if mel.shape[-1] < 128:
        mel = F.pad(mel, (0, 128 - mel.shape[-1]))
    else:
        mel = mel[:, :, :128]

    mel = (mel - mel.mean()) / (mel.std() + 1e-6)
    mel = mel.unsqueeze(0).to(device)

    # ----- MODEL -----
    img_emb = image_encoder(img)
    aud_emb = audio_encoder(mel)

    out = fusion_model(img_emb, aud_emb)
    probs = torch.softmax(out, dim=1)[0].cpu().numpy()
    pred = probs.argmax()

    return pred, probs


In [11]:
labels = [
    "neutral", "calm", "happy", "sad",
    "angry", "fearful", "disgust", "surprised"
]


In [23]:
video_path = "test_video3.mp4"  # your uploaded video

pred, probs = predict_from_video(video_path)

print("\nPredicted Emotion:", labels[pred])
print("Confidence scores:")
for l, p in zip(labels, probs):
    print(f"{l:10s}: {p:.3f}")



Predicted Emotion: calm
Confidence scores:
neutral   : 0.002
calm      : 0.800
happy     : 0.009
sad       : 0.076
angry     : 0.013
fearful   : 0.005
disgust   : 0.092
surprised : 0.004
